## About this notebook:

This notebook extracts information from single cells in the 4i experiment. To customize calculated parameters read documentation of scikit-image regionprops: https://scikit-image.org/docs/dev/api/skimage.measure.html#skimage.measure.regionprops.

Input:
- data frames for selected wells
- labels
- selected round (anchor of alignment)
- punchmask (mask of excluded regions)
- aligned tiffs

Output:
- data frames with features of nuclei and rings

## Fill in info about the experiment to process

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from project_config import load_config

config    = load_config()
base_dir  = config['base_dir']
well_list = config['well_list']

# Sub-folder paths
path_df     = os.path.join(base_dir, 'output_df')
path_labels = os.path.join(base_dir, 'segmented')
path_mask   = os.path.join(base_dir, 'masks')
path_tiffs  = os.path.join(base_dir, 'aligned_tiffs')
path_save   = os.path.join(base_dir, 'cell_data')

label_round = 0  # anchor round, 0-indexed

In [3]:
# settings for properties to calculate
properties = ['label', 'area','centroid','orientation','major_axis_length','minor_axis_length','bbox','mean_intensity']
properties_ring = ['label','area', 'mean_intensity']

## Prepare for processing

In [5]:
import os
import pickle
import pickle as pickle
import numpy as np
import pandas as pd
from tifffile import imread
from skimage.measure import regionprops_table
from skimage.segmentation import clear_border
from skimage.draw import polygon
from ring_functions import make_rings

In [6]:
# create directory for saving data frames if needed
try:
    os.mkdir(path_save)
    print('Directory for saving cell data created.')
except:
    print('Directory not created.')

Directory for saving cell data created.


## Process

In [ ]:
for selected_well in well_list:
    
    print(f'Processing well {selected_well}.')
    
    # open data frame
    df_name = f'df_{selected_well}.pkl'
    df = pd.read_pickle(os.path.join(path_df,df_name))
    
    ##############################################################
    
    # open labels
    print('Loading labels.')
    labels_path = os.path.join(path_labels,selected_well,f'labels_{selected_well}_round_{str(label_round).zfill(3)}.tif')
    labels = imread(labels_path)
    
    # clear border objects
    labels = clear_border(labels)
    
    ##############################################################
    
    # open punchmask
    print('Loading mask.')
    
    # translate shapes into an image
    mask = np.zeros(labels.shape).astype(int)
    
    try:
        masking = os.path.join(path_mask,f'mask_{selected_well}.pkl')
        sh = pickle.load( open( masking, "rb" ) )

        for reg in sh:
            rr, cc = polygon(reg[:,0],reg[:,1]) 
            mask[rr,cc] = 1
            
        print('Existing mask found.')
        
    except:
        print('Mask not found.')
        
    
    ##############################################################
    
    # open intensity images
    print('Loading intensity images.')
    int_im = []
    for ind,signal in df.iterrows():
        
        im_path = os.path.join(path_tiffs,selected_well,signal.aligned_file_name)
        im = imread(im_path)
        
        int_im.append(im)
        
    int_im.append(mask)    
        
    int_im = np.array(int_im)
    int_im = np.moveaxis(int_im,0,2)

        
    ##############################################################
        
    # calculate properties of nuclei
    print('Quantifying nuclei.')
    nuclei_df = pd.DataFrame(regionprops_table(labels, properties=properties, intensity_image=int_im))
    nuclei_df = nuclei_df.rename(columns={'area': 'nuc_area'})

    # adjust names
    channel_list = ['_'.join([str(x).zfill(2), y]) for x, y in zip(df.nameRound, df.signal.to_list())]
    df['round_channel'] = channel_list

    intensity_names = [f'{x}_nuc_mean' for x in df.round_channel]
    intensity_names.append('nuc_mask')

    # Set the column names of nuclei_df
    nuclei_df.columns = list(nuclei_df.columns[:-len(df) - 1]) + intensity_names
    
    ##############################################################
        
    # generate rings
    print('Generating cytoplasmic rings.')
    rings = make_rings(labels,width=6,gap=1)
    
    # calculate properties of rings
    print('Quantifying cytoplasmic rings.')
    rings_df = pd.DataFrame(regionprops_table(rings, properties=properties_ring, intensity_image=int_im))
    rings_df = rings_df.rename(columns={'area': 'ring_area'})
    
    # adjust names
    intensity_names = [f'{x}_ring_mean' for x in df.round_channel]
    intensity_names.append('ring_mask')
    
    rings_df.columns=list(rings_df.columns[:-len(df)-1])+intensity_names

    ##############################################################
        
    # merging data frames
    cell_data = pd.merge(nuclei_df,rings_df,how='inner',on='label',suffixes=('_nuc', '_ring'))

    #trim out rows without DNA
    for column in cell_data.columns:
        if column.endswith('_DNA_nuc_mean'):
            cell_data = cell_data[cell_data[column] != 0]

    #Calculate protein and DNA totals for Nucleus and Rings
    # Get the columns containing 'nuc_mean' in their names
    nuc_mean_columns = [col for col in cell_data.columns if 'nuc_mean' in col]
    ring_mean_columns = [col for col in cell_data.columns if 'ring_mean' in col]

    # Iterate over each nuc_mean column
    for nuc_mean_col in nuc_mean_columns:
        # Extract the common prefix
        prefix = nuc_mean_col.split('_nuc_mean')[0]

        # Multiply each cell in the nuc_mean column by the 'nuc_area' and store the result in total_protein column
        cell_data[f"{prefix}_total_nuc_protein"] = cell_data[nuc_mean_col] * cell_data['nuc_area']


    for nuc_col, ring_col in zip(nuc_mean_columns, ring_mean_columns):
        # Extract the common prefix
        prefix = nuc_col.split('_nuc_mean')[0]
        suffix = ring_col.split('_ring_mean')[0]

        # Multiply each cell in the ring_mean column by the 'nuc_area' and store the result in total_protein column
        cell_data[f"{prefix}_nuc_cyto_ratio"] = cell_data[nuc_col] / cell_data[ring_col]


    ##############################################################
        
    # add info and save 
    print("Ready to Save")
    cell_data['well'] = selected_well
XSD 
    cell_data.to_pickle(os.path.join(path_save,f'cell_data_{selected_well}_df.pkl'),protocol=4)
    cell_data.to_csv(os.path.join(path_save,f'cell_data_{selected_well}_df.csv'))
    

Processing well B02.
Loading labels.
Loading mask.
Existing mask found.
Loading intensity images.
Quantifying nuclei.
Generating cytoplasmic rings.
Quantifying cytoplasmic rings.
Ready to Save
Processing well B03.
Loading labels.
Loading mask.
Existing mask found.
Loading intensity images.
Quantifying nuclei.
Generating cytoplasmic rings.
Quantifying cytoplasmic rings.
Ready to Save
Processing well B04.
Loading labels.
Loading mask.
Existing mask found.
Loading intensity images.
Quantifying nuclei.
Generating cytoplasmic rings.
Quantifying cytoplasmic rings.
Ready to Save
Processing well B05.
Loading labels.
Loading mask.
Existing mask found.
Loading intensity images.
Quantifying nuclei.
Generating cytoplasmic rings.
Quantifying cytoplasmic rings.
Ready to Save
Processing well B06.
Loading labels.
Loading mask.
Existing mask found.
Loading intensity images.
Quantifying nuclei.
Generating cytoplasmic rings.
Quantifying cytoplasmic rings.
Ready to Save
Processing well B07.
Loading labels

: 

In [7]:
path_save

'Z:\\Clara\\20260304_Exp 1.2\\cell_data'

# concatenate all of the .csv files into a sinlge file with book keeping parameters

In [8]:
import glob


In [ ]:

# generate a list of all of the .csv file
csv_list = glob.glob(os.path.join(path_save, '*.csv'))

# generate a list of all of the well names
well_id_list = [file_iter.split('_df')[0].split('\\cell_data_')[1] for file_iter in csv_list]
well_id_list

['D02',
 'D03',
 'D04',
 'D05',
 'D06',
 'D07',
 'E02',
 'E03',
 'E04',
 'E05',
 'E06',
 'E07']

In [17]:
df_ = pd.read_csv(csv_list[0])
df_.well[0]

'D02'

In [21]:
time_dict = {'D02' : 2,
             'D03' :2,
            'D04':2,
            'D05':2,
            'D06':2,
            'D07':2,
            'E02':1,
            'E03':1,
            'E04':1,
            'E05':1,
            'E06':1,
            'E07':1}

palbo_conc_dict = {'D02' : 0,
                   'D03' : 0,
                'D04': 0,
                'D05': 100,
                'D06': 100,
                'D07': 100,
                'E02': 0,
                'E03':0,
                'E04':0 ,
                'E05': 100,
                'E06':100,
                'E07': 100}
# list to hold modified
df_lst = []

# iterate over the .csv files, add a couple of book keeping parameters and concatenate together
for csv_iter in csv_list:
    df_ = pd.read_csv(csv_iter)

    well_idx = df_.well[0]

    df_['treatment_time'] = time_dict[well_idx]
    df_['palbo_conc'] = palbo_conc_dict[well_idx]


    df_lst.append(df_)

df_out = pd.concat(df_lst, ignore_index=True)

out_file = os.path.join(path_save, 'all_wells_concatted26030901.csv')

df_out.to_csv(out_file)
                        

                        


In [20]:
df_

,Unnamed: 0,label,nuc_area,centroid-0,centroid-1,orientation,major_axis_length,minor_axis_length,bbox-0,bbox-1,...,R3_DNA_nuc_cyto_ratio,R3_CyclinD1_nuc_cyto_ratio,R3_CyclinE1_nuc_cyto_ratio,R3_p21_nuc_cyto_ratio,R4_DNA_nuc_cyto_ratio,R4_cMyc_nuc_cyto_ratio,R4_EdU_nuc_cyto_ratio,well,treatment_time,palbo_conc
0,0,68,483,12.788820,680.561077,0.808268,29.050448,21.302759,1,668,...,3.045264,1.148583,1.113794,1.043580,3.912675,2.020215,2.486854,D02,2,0
1,1,69,378,10.851852,3315.010582,1.155748,23.044741,20.960397,1,3304,...,2.602250,1.304419,1.249938,1.755656,2.497771,1.181022,0.364064,D02,2,0
2,2,70,496,12.727823,5700.786290,-1.030396,28.617239,22.089455,1,5688,...,4.891695,1.629612,1.325786,1.452755,5.365381,1.973594,1.020274,D02,2,0
3,3,71,1162,19.793460,6181.517212,-0.781750,39.412780,37.642634,1,6163,...,1.527943,1.073351,1.348405,1.118744,1.521643,1.137256,1.709595,D02,2,0
4,4,72,513,11.781676,2865.485380,-1.444798,31.140872,21.052949,2,2851,...,4.566573,1.576585,1.295731,1.200960,4.813631,1.273354,1.000151,D02,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27908,28367,29984,504,6851.515873,1944.440476,-1.104707,29.655076,21.654022,6840,1931,...,1.158304,0.478669,0.471011,0.487225,5.100518,1.966896,1.158342,D02,2,0
27909,28368,29985,434,6850.417051,2230.154378,1.165458,27.750155,19.996306,6840,2218,...,0.870371,0.399477,0.315113,0.310517,2.433338,0.810952,0.709260,D02,2,0
27910,28370,29987,342,6849.967836,1822.321637,1.294032,21.713271,20.141810,6841,1812,...,4.476023,1.826472,1.872566,1.593229,7.475595,3.034706,1.797608,D02,2,0
27911,28371,29988,425,6852.305882,2686.014118,-0.726880,24.591642,22.060033,6841,2675,...,3.421929,2.179458,2.419038,2.402722,4.098486,2.069124,1.902549,D02,2,0
